---
title: "Deep Learning Image Upscaling: Transforming Low-Resolution to High-Resolution Images"
author: "Fabian"
date: "2024-01-24"
categories: [GenAI]
format:
  html:
    code-fold: false
    toc: true
jupyter: python3
draft: False
---

# Deep Learning Image Upscaling: Transforming Low-Resolution to High-Resolution Images

## Introduction to Super-Resolution

Image upscaling, or super-resolution, is one of the most fascinating challenges in computer vision and deep learning. Imagine taking a blurry, pixelated image and magically transforming it into a crisp, detailed masterpiece. This isn't science fiction—it's the power of deep learning, specifically convolutional neural networks (CNNs) designed for image super-resolution.

### The Problem with Traditional Upscaling

Before deep learning, image upscaling was a mundane task performed by simple interpolation algorithms:
- Nearest neighbor: Duplicates pixels, creating a blocky result
- Bilinear interpolation: Averages surrounding pixels, slightly smoother but still low quality
- Bicubic interpolation: More sophisticated averaging, better but still limited

These traditional methods fundamentally lack intelligence. They can't "imagine" or "hallucinate" details that weren't in the original image.

## Deep Learning: A Paradigm Shift in Image Upscaling

Enter deep learning—specifically, convolutional neural networks that can learn to reconstruct high-frequency details from low-resolution inputs.


![CAT image we will upscale](cat.png)


### The Core Architecture: Residual Learning

My PyTorch implementation uses a residual learning approach, which is crucial for effective super-resolution. Let's break down why this matters:

1. **Residual Blocks**: These are the heart of our network. Each block learns to capture and reconstruct image details incrementally.


In [1]:
import torch
import torchvision.transforms as transforms
import torch.nn as nn
import numpy as np
from PIL import Image
import os

In [2]:
torch.cuda.empty_cache()

In [ ]:


class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.prelu = nn.PReLU()
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.prelu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += residual
        return out

class SuperResolutionModel(nn.Module):
    def __init__(self, scale_factor=2):
        super(SuperResolutionModel, self).__init__()
        self.initial_conv = nn.Conv2d(3, 64, kernel_size=9, padding=4)
        self.prelu = nn.PReLU()
        
        self.residual_blocks = nn.Sequential(
            *[ResidualBlock(64) for _ in range(16)]
        )
        
        self.bottleneck = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        
        self.upscale = nn.Sequential(
            nn.Conv2d(64, 256, kernel_size=3, padding=1),
            nn.PixelShuffle(2),
            nn.PReLU(),
            nn.Conv2d(64, 3, kernel_size=9, padding=4)
        )

    def forward(self, x):
        initial = self.initial_conv(x)
        x = self.prelu(initial)
        x = self.residual_blocks(x)
        x = self.bottleneck(x)
        x += initial
        x = self.upscale(x)
        return x

class ImageUpscaler:
    def __init__(self, model_path=None, scale_factor=2):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = SuperResolutionModel(scale_factor).to(self.device)
        
        if model_path and os.path.exists(model_path):
            self.model.load_state_dict(torch.load(model_path, map_location=self.device))
        
        self.model.eval()

    def preprocess_image(self, image_path):
        image = Image.open(image_path).convert('RGB')
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                 std=[0.229, 0.224, 0.225])
        ])
        return transform(image).unsqueeze(0).to(self.device)

    def upscale(self, input_path, output_path):
        with torch.no_grad():
            input_tensor = self.preprocess_image(input_path)
            upscaled = self.model(input_tensor)
            
            # Denormalize and convert back to image
            upscaled = upscaled.squeeze(0).clamp(0, 1)
            upscaled = transforms.ToPILImage()(upscaled.cpu())
            upscaled.save(output_path)
        return output_path

    def batch_upscale(self, input_dir, output_dir):
        os.makedirs(output_dir, exist_ok=True)
        for filename in os.listdir(input_dir):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
                input_path = os.path.join(input_dir, filename)
                output_path = os.path.join(output_dir, f'upscaled_{filename}')
                self.upscale(input_path, output_path)


In [8]:

upscaler = ImageUpscaler()
    
    # Single image upscaling
upscaler.upscale('cat.png', 'upscaled_image.png')
    
    # Batch upscaling
#upscaler.batch_upscale('input_images/', 'upscaled_images/')

In [4]:

class MemoryEfficientResidualBlock(nn.Module):
    def __init__(self, channels):
        super(MemoryEfficientResidualBlock, self).__init__()
        # Reduced complexity to save memory
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.prelu = nn.PReLU()
        
    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.prelu(out)
        out += identity
        return out

class LightweightSuperResolutionModel(nn.Module):
    def __init__(self, scale_factor=2, base_channels=32):
        super(LightweightSuperResolutionModel, self).__init__()
        # Reduced model complexity
        self.initial_conv = nn.Conv2d(3, base_channels, kernel_size=3, padding=1)
        self.prelu = nn.PReLU()
        
        # Fewer residual blocks to reduce memory footprint
        self.residual_blocks = nn.Sequential(
            *[MemoryEfficientResidualBlock(base_channels) for _ in range(8)]
        )
        
        self.upscale = nn.Sequential(
            nn.Conv2d(base_channels, base_channels * 4, kernel_size=3, padding=1),
            nn.PixelShuffle(2),
            nn.PReLU(),
            nn.Conv2d(base_channels, 3, kernel_size=3, padding=1)
        )

    def forward(self, x):
        x = self.initial_conv(x)
        x = self.prelu(x)
        x = self.residual_blocks(x)
        x = self.upscale(x)
        return x

class MemoryEfficientImageUpscaler:
    def __init__(self, scale_factor=2):
        # Detect best available device with memory considerations
        self.device = self._select_optimal_device()
        
        # Lightweight model initialization
        self.model = LightweightSuperResolutionModel(scale_factor).to(self.device)
        self.model.eval()

    def _select_optimal_device(self):
        """
        Intelligently select the most memory-efficient device
        Prioritizes MPS (Metal Performance Shaders) for MacBooks
        """
        if torch.backends.mps.is_available():
            return torch.device('mps')
        elif torch.cuda.is_available():
            # Use smallest GPU if multiple are available
            return torch.device('cuda:0')
        return torch.device('cpu')

    def preprocess_image(self, image_path, max_size=1024):
        """
        Memory-efficient image preprocessing with size limitation
        """
        image = Image.open(image_path).convert('RGB')
        
        # Resize large images to reduce memory consumption
        width, height = image.size
        if max(width, height) > max_size:
            scaling_factor = max_size / max(width, height)
            new_width = int(width * scaling_factor)
            new_height = int(height * scaling_factor)
            image = image.resize((new_width, new_height), Image.LANCZOS)
        
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                 std=[0.229, 0.224, 0.225])
        ])
        return transform(image).unsqueeze(0).to(self.device)

    def upscale(self, input_path, output_path, cleanup_intermediate=True):
        """
        Memory-efficient upscaling with optional intermediate cleanup
        """
        try:
            with torch.no_grad():
                # Use torch.inference_mode for even lower memory overhead
                with torch.inference_mode():
                    input_tensor = self.preprocess_image(input_path)
                    
                    # Free up memory after preprocessing
                    torch.cuda.empty_cache()
                    
                    upscaled = self.model(input_tensor)
                    
                    # Denormalize and convert back to image
                    upscaled = upscaled.squeeze(0).clamp(0, 1)
                    upscaled_image = transforms.ToPILImage()(upscaled.cpu())
                    upscaled_image.save(output_path)
        except RuntimeError as e:
            print(f"Memory error during upscaling: {e}")
            print("Attempting to upscale with CPU fallback...")
            return self._cpu_fallback_upscale(input_path, output_path)
        
        # Optional cleanup to free memory
        if cleanup_intermediate:
            del input_tensor, upscaled
            torch.cuda.empty_cache()
        
        return output_path

    def _cpu_fallback_upscale(self, input_path, output_path):
        """
        Fallback method for when GPU memory is insufficient"""
        with torch.device('cpu'):
            self.model = self.model.to('cpu')
            return self.upscale(input_path, output_path)

    def batch_upscale(self, input_dir, output_dir, max_batch_size=4):
        """
        Batch processing with controlled memory usage
        """
        os.makedirs(output_dir, exist_ok=True)
        image_files = [f for f in os.listdir(input_dir) 
                       if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif'))]
        
        # Process in smaller batches to manage memory
        for i in range(0, len(image_files), max_batch_size):
            batch = image_files[i:i+max_batch_size]
            
            for filename in batch:
                input_path = os.path.join(input_dir, filename)
                output_path = os.path.join(output_dir, f'upscaled_{filename}')
                self.upscale(input_path, output_path)
            
            # Force garbage collection and clear CUDA cache
            torch.cuda.empty_cache()


In [ ]:
upscaler = MemoryEfficientImageUpscaler()
    # Single image upscaling
upscaler.upscale('cat.png', 'upscaled_image.png')

#### Why Residual Blocks Work

- They solve the vanishing gradient problem in deep networks
- Allow networks to learn incremental image details
- Preserve original image information while adding refinements

### Upscaling Mechanism: From Low to High Resolution

Our network uses a clever upscaling technique called `PixelShuffle`, which reorganizes low-resolution feature maps into a higher resolution output.

In [ ]:
self.upscale = nn.Sequential(
    nn.Conv2d(64, 256, kernel_size=3, padding=1),
    nn.PixelShuffle(2),  # Magic upscaling layer
    nn.PReLU(),
    nn.Conv2d(64, 3, kernel_size=9, padding=4)
)







#### How PixelShuffle Works
1. Learns compact, low-resolution feature representations
2. Efficiently rearranges these features into a higher resolution grid
3. Introduces learned, context-aware upscaling instead of fixed interpolation

### Training Considerations

While our example doesn't include training code, effective super-resolution models typically:
- Use perceptual loss functions (like VGG-based loss)
- Train on diverse, high-quality image datasets
- Employ data augmentation to improve generalization

## Performance and Limitations

### Strengths
- Can recover seemingly "lost" image details
- Works across various image types
- GPU-accelerated for real-time performance

### Limitations
- Quality depends heavily on training data
- Cannot create details that are completely absent
- Computational complexity increases with model depth

## Practical Implementation Tips for PyTorch Engineers

1. Always use GPU acceleration when possible
2. Experiment with different residual block configurations
3. Normalize input images (as we do in the preprocessing step)
4. Consider transfer learning from pre-trained models

## Conclusion

Deep learning-based image upscaling represents a convergence of signal processing, computer vision, and neural network design. By understanding the underlying mechanisms—residual learning, intelligent feature reconstruction, and efficient upscaling techniques—we can create models that transform how we perceive and process digital imagery.

### Next Steps for Learning

- Experiment with different network architectures
- Study state-of-the-art models like ESRGAN and EDSR
- Collect and prepare diverse training datasets

Happy upscaling, AI engineers! 🚀📸